# 23_tokenizer_hf.ipynb

**12주차 · 1교시** 실습 노트북

- 이론 설명과 관찰 포인트는 배포 자료(`12week/student/`)를 함께 보세요.
- 실행 환경: `%DL2026_HOME%\venv` 활성화 후 `Python (dl2026)` 커널.
- 전체 10셀. 위에서부터 순서대로 실행합니다.

## 3. 실습 1 — 토크나이저 관찰 ★

**셀 1** — 토크나이저 로딩

In [ ]:
from transformers import AutoTokenizer

MODEL = "klue/bert-base"                       # ★ 수업에서 쓸 모델 (배포 캐시)
tok = AutoTokenizer.from_pretrained(MODEL)

print("어휘 크기 :", tok.vocab_size)
print("최대 길이 :", tok.model_max_length)
print("특수 토큰 :", tok.all_special_tokens)

**셀 2** — 한국어가 어떻게 쪼개지나 ★

In [ ]:
texts = [
    "이 영화 정말 재미없었다",
    "연출은 좋았는데 각본이 아쉬웠음",
    "핵노잼 ㅋㅋㅋ 시간낭비",              # 신조어·자모
    "OST가 진짜 미쳤다 goat",              # 영어 혼용
    "스토리는뭐랄까그냥그랬어요",          # 띄어쓰기 없음
]
for t in texts:
    pieces = tok.tokenize(t)
    print(f"{t}\n   → {pieces}   ({len(pieces)} 토큰)\n")

> **관찰 포인트 ★★**: ① 흔한 말은 통째로, 드문 말은 잘게 쪼개집니다. ② `##` 이 붙은 조각은 **앞 조각에 이어진다**는 뜻입니다. ③ 띄어쓰기가 없어도 **어느 정도 복원**됩니다. ④ **`[UNK]` 가 거의 안 나옵니다** — 이것이 서브워드의 힘입니다.

**셀 3** — 특수 토큰과 인코딩 결과 ★

In [ ]:
enc = tok("이 영화 정말 재미없었다")
print("키 :", list(enc.keys()))
print("input_ids      :", enc["input_ids"])
print("attention_mask :", enc["attention_mask"])
print("\n토큰으로 되돌리면 :", tok.convert_ids_to_tokens(enc["input_ids"]))

**셀 4** — 패딩·자르기 · attention_mask ★

In [ ]:
batch = tok(["짧다", "이 영화는 정말 길고 지루하고 재미없고 실망스러웠다"],
            padding=True, truncation=True, max_length=16, return_tensors="pt")

print("input_ids :\n", batch["input_ids"])
print("\nattention_mask :\n", batch["attention_mask"])
print("\nshape :", batch["input_ids"].shape)      # (2, T)

> **핵심 ★ (출제 지점)**: **`attention_mask` 는 패딩 자리를 알려 주는 표시**입니다. 1 인 자리만 어텐션에 참여하고, 0 인 자리는 **11주차에 배운 그 마스킹**으로 제외됩니다. *"11주차에 `-inf` 를 채우던 그 마스크가 여기 이 배열입니다."*

**셀 5** — decode 왕복 · 정보 손실 확인

In [ ]:
s = "OST가 진짜 미쳤다 goat"
ids = tok(s)["input_ids"]
print("원문        :", s)
print("복원        :", tok.decode(ids))
print("특수 토큰 뺀 복원 :", tok.decode(ids, skip_special_tokens=True))

> **관찰 포인트**: 복원이 **완벽하지 않을 수 있습니다**(띄어쓰기·대소문자). 토큰화는 **되돌릴 수 있지만 무손실은 아닙니다.**

**셀 6** — 내 문장 넣어 보기 (자유)

In [ ]:
my = input("문장을 입력하세요 : ")
print(tok.tokenize(my))

## 4. 실습 2 — `datasets` 로 NSMC 로딩

**셀 7** — 데이터 로딩

In [ ]:
from datasets import load_dataset

try:
    raw = load_dataset("nsmc")                      # Hub 에서
except Exception as e:
    print("Hub 실패 → 배포 파일 사용:", e)
    raw = load_dataset("csv", data_files={          # ★ 배포한 CSV 로 대체
        "train": "data/nsmc_subset/train.csv",
        "test":  "data/nsmc_subset/test.csv"})

print(raw)
print("\n샘플 :", raw["train"][0])

**셀 8** — 서브셋 추출 ★ 시간 관리의 핵심

In [ ]:
N_TRAIN, N_TEST = 20000, 4000                       # ★ 고정. 값이 다르면 비교 불가
train = raw["train"].shuffle(seed=42).select(range(N_TRAIN))
test  = raw["test"].shuffle(seed=42).select(range(N_TEST))

print("train :", len(train), "| test :", len(test))
print("레이블 분포 :", {l: train['label'].count(l) for l in set(train['label'])})

> **핵심 ★**: 전체 20만 건을 쓰면 8GB GPU 에서 **30분 이상** 걸려 수업 시간 안에 끝나지 않습니다. **2만 건 서브셋**으로도 85~89% 는 나옵니다. *"현실의 제약 안에서 실험을 설계하는 것"* 자체가 이 과목이 가르치는 것입니다. `seed=42` 로 **재현성**도 확보합니다(6주차).

**셀 9** — map 으로 한 번에 토크나이즈 ★

In [ ]:
MAX_LEN = 64                                        # ★ 고정

def tokenize_fn(batch):
    return tok(batch["document"], truncation=True, max_length=MAX_LEN)

train_tok = train.map(tokenize_fn, batched=True)     # batched=True → 훨씬 빠르다
test_tok  = test.map(tokenize_fn,  batched=True)

print(train_tok)
print("\n첫 샘플 키 :", list(train_tok[0].keys()))
print("input_ids 앞 12개 :", train_tok[0]["input_ids"][:12])

**셀 10** — max_length 를 정하는 근거 ★

In [ ]:
import numpy as np
lens = [len(x) for x in tok(train["document"][:2000], truncation=False)["input_ids"]]
print(f"토큰 길이  평균 {np.mean(lens):.1f} | 중앙값 {np.median(lens):.0f} "
      f"| 95% {np.percentile(lens,95):.0f} | 최대 {max(lens)}")
print(f"\nmax_length=64 로 자르면 {np.mean(np.array(lens) > 64)*100:.1f}% 의 리뷰가 잘린다")

> **핵심 ★★ (기말 출제 지점)**: `max_length` 를 줄이면 **얻는 것** = 메모리·시간(어텐션은 길이의 제곱), **잃는 것** = 뒷부분 정보. 감성은 문장 끝에 오는 경우가 많아 **자르는 위치가 성능에 영향**을 줍니다(10주차와 같은 논점). **숫자를 보고 정하는 습관**을 들이세요 — 95 백분위수를 기준으로 잡는 것이 실무 관행입니다.